# WISER Module 4 — Making Meaning with NLP Semantics

This notebook supports **Rise Sections 4.3–4.4**. Your main product is a documented terminology crosswalk. Work through the steps in order: **ingest → check de-identification → normalize → recognize entities → review concept links → audit language → export and reflect**.

**Start here:** In Colab, select **File → Save a copy in Drive**, then **Runtime → Run all**. Read each result, edit the marked learner cells, and rerun those cells and the export. Running all without editing leaves an incomplete assignment. Use only the provided synthetic clinical notes; do not paste real notes or PHI.

**Choose one pathway:** Foundational (prepared code, three crosswalk rows, 200-word reflection) **or** Applied (your own ingestion and transformer NER, embedding-neighbor audit, documented bundle and 500-word validity memo). Rise does not require both. Complete the matching section below and follow your instructor's submission directions.

In [ ]:
PATHWAY = "Foundational"  # Change to "Applied" only if you selected 4.32 in Rise.
assert PATHWAY in {"Foundational", "Applied"}
print("Pathway:", PATHWAY)

## 1. Ingest synthetic data (Rise 4.4, step 1)

The repository's `notes.csv` contains 2,200 rows but only 10 distinct note texts. We use one representative of each distinct text for the practice pipeline and display both counts. Repeated identical text should not be interpreted as independent evidence. The original dataset remains in `all_notes` for inspecting its fields.

In [ ]:
from pathlib import Path
import json, re, shutil
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
OUT = Path("module4_outputs")
OUT.mkdir(exist_ok=True)
DATA_URL = "https://raw.githubusercontent.com/mcbacloudlab/WISERTraining-Modules-/refs/heads/main/Module%204/Data/notes.csv"
all_notes = pd.read_csv(DATA_URL)
required = {"note_id", "text", "deidentified"}
if not required.issubset(all_notes.columns):
    raise ValueError(f"Missing required fields: {sorted(required - set(all_notes.columns))}")
all_notes["text"] = all_notes["text"].fillna("").astype(str)
notes = (all_notes.sort_values("note_id").drop_duplicates("text").reset_index(drop=True).copy())
print(f"Loaded {len(all_notes):,} rows; {len(notes)} distinct texts for the practice pipeline.")
display(notes[["note_id", "author_role", "language_context", "text", "key_terms_legacy", "key_terms_current"]].head(10))

## 2. Inspect for identifiers (Rise 4.4, step 2)

The file has a `deidentified` indicator. The simple pattern check below looks for some obvious identifier shapes in the text. **It cannot prove that text is safe to share**. Do not add real clinical notes. Read the results and explain what a human reviewer would still need to check.

In [ ]:
identifier_patterns = {
    "email-like": r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}",
    "phone-like": r"(?<!\d)\d{3}[-. ]\d{3}[-. ]\d{4}(?!\d)",
    "MRN-like": r"\bMRN[: #]*\d+\b",
}
def identifier_flags(text):
    return [label for label, pattern in identifier_patterns.items() if re.search(pattern, str(text), re.I)]
notes["identifier_flags"] = notes["text"].map(identifier_flags)
print("deidentified values:", all_notes["deidentified"].value_counts(dropna=False).to_dict())
print("Distinct note texts flagged by this limited check:", int(notes["identifier_flags"].str.len().gt(0).sum()))
display(notes[["note_id", "identifier_flags"]])

In [ ]:
deid_response = """[Confirm what you observed. Why does this check not replace human review?]"""
print(deid_response)

## 3. Normalize wording (Rise 4.31, task 2)

**Your edit:** Add one abbreviation to `abbreviation_rules` and one drug-name variation to `drug_variant_rules`. The examples show the format. Review the before/after output; normalization is different from de-identification. Do not silently rewrite the protected original text.

In [ ]:
abbreviation_rules = {
    "moud": "medications for opioid use disorder",  # example
    # TODO: add "oud": "opioid use disorder"
}
drug_variant_rules = {
    "fent": "fentanyl",  # example; whole-word matching prevents replacing parts of other words
    # TODO: add another documented variant used in your teaching example
}
def normalize_text(text):
    result = str(text).lower()
    for old, new in {**abbreviation_rules, **drug_variant_rules}.items():
        result = re.sub(r"(?<!\w)" + re.escape(old) + r"(?!\w)", new, result)
    result = re.sub(r"\s+", " ", result).strip()
    return result
practice_phrases = ["Person asks about MOUD for OUD.", "Person reports fent exposure."]
display(pd.DataFrame({"original": practice_phrases, "normalized": [normalize_text(x) for x in practice_phrases]}))
notes["normalized_text"] = notes["text"].map(normalize_text)
if len(abbreviation_rules) < 2 or len(drug_variant_rules) < 2:
    print("TO COMPLETE: add one abbreviation rule and one drug-name variant rule above.")
display(notes[["note_id", "text", "normalized_text"]].head())

In [ ]:
normalization_response = """[Name both rules you added and explain why each preserves the intended meaning.]"""
print(normalization_response)

## 4. Recognize mentions with provided rules (Rise 4.31, task 3)

**NER** locates and labels spans of text. It does not establish which controlled-vocabulary concept a mention represents. Run the provided patterns for drug, dose, and route; a separate disorder pattern gives extra context. Dose and route may be absent from these ten texts, so the practice phrases demonstrate how the recognizer behaves. Review one missed or potentially incorrect mention.

In [ ]:
patterns = {
    "drug": r"\b(?:methadone|buprenorphine|fentanyl|naloxone|kratom|opioids?)\b",
    "dose": r"\b\d+(?:\.\d+)?\s*(?:mg|mcg|ml)\b",
    "route": r"\b(?:oral|sublingual|intranasal|intravenous)\b",
    "disorder": r"\bopioid use disorder\b",
}
def recognize(text):
    return {label: sorted(set(re.findall(pattern, str(text), flags=re.I)))
            for label, pattern in patterns.items()}
practice_ner = ["5 mg oral methadone discussed.", "Intranasal naloxone was offered."]
display(pd.DataFrame({"practice_text": practice_ner, "recognized": [recognize(x) for x in practice_ner]}))
notes["entities"] = notes["normalized_text"].map(recognize)
for label in patterns:
    notes[label + "_mentions"] = notes["entities"].map(lambda d, key=label: "; ".join(d[key]))
display(notes[["note_id", "drug_mentions", "dose_mentions", "route_mentions", "disorder_mentions"]])

In [ ]:
ner_response = """[Choose one note or practice phrase. Which mention was found? What did the rule miss or potentially mislabel?]"""
print(ner_response)

## 5. Link concepts and build the terminology crosswalk (Rise 4.31, task 4)

**Recognition is not linking.** A medication mention can be considered for **RxNorm**; a survey question can be compared with a relevant **HEAL CDE**. A similar label is not enough to claim equivalence: compare definitions, question wording, timeframes, response options, and source years. Mark a row `supported`, `uncertain`, or `no suitable mapping`. Leave the target ID blank if you cannot verify it; never invent a code.

**Foundational task:** Fill **three rows** using the three-wave NSDUH teaching subset and codebooks provided in Rise. Enter the actual source variable and year, original wording, mapping decision, rationale, and a specific source/codebook reference for each. The `notes.csv` fields `key_terms_legacy` and `key_terms_current` are prompts for investigation, **not codebook evidence**. The example display below shows where legacy wording occurs; it is not a verified NSDUH crosswalk.

In [ ]:
display(notes.loc[notes["key_terms_legacy"].notna(),
    ["note_id", "text", "key_terms_legacy", "key_terms_current"]])
CROSSWALK_COLUMNS = [
    "mapping_id", "source_dataset", "source_year_or_version", "source_variable_or_record_id",
    "original_wording", "source_definition_or_context", "source_availability",
    "target_system", "target_id_or_variable", "target_label_and_definition",
    "mapping_status", "mapping_rationale", "source_reference", "target_reference", "crosswalk_version"
]
# Replace the bracketed entries with evidence from the Rise teaching subset/codebooks.
# Use an empty target ID and explain uncertainty if you cannot verify a match.
crosswalk_rows = []
for number in range(1, 4):
    crosswalk_rows.append({
        "mapping_id": f"CW-{number:03d}",
        "source_dataset": "[NSDUH teaching subset / exact source]",
        "source_year_or_version": "[year]",
        "source_variable_or_record_id": "[variable name]",
        "original_wording": "[exact source wording]",
        "source_definition_or_context": "[definition and timeframe]",
        "source_availability": "[which waves asked this item?]",
        "target_system": "[HEAL CDE / RxNorm / none]",
        "target_id_or_variable": "",  # Enter only a verified ID; blank is acceptable for uncertain/unmapped rows.
        "target_label_and_definition": "[verified target definition, if any]",
        "mapping_status": "[supported / uncertain / no suitable mapping]",
        "mapping_rationale": "[why this decision follows from the definitions]",
        "source_reference": "[specific NSDUH codebook and page/item]",
        "target_reference": "[verified source for target, or not applicable]",
        "crosswalk_version": "draft-01",
    })
crosswalk = pd.DataFrame(crosswalk_rows, columns=CROSSWALK_COLUMNS)
display(crosswalk.T)

**Decision check:** Did a question appear in one wave but not another? Record the earlier wave as **not asked** in `source_availability`; do not backfill it. A wording change may be uncertain rather than equivalent. If you use a medication mention, use a verified RxNorm concept only when the note supports that level of specificity.

In [ ]:
mapping_response = """[In 2–4 sentences, explain one supported, uncertain, or unmapped decision using the source definitions.]"""
print(mapping_response)

## 6. Audit language and bias (Rise 4.31, task 5)

The helper flags candidate wording in the original synthetic text. The original remains available in memory for review; a separately labeled analysis field can carry a reviewed revision. **Your edit:** Write a respectful **complete replacement sentence** for the flagged note and explain the choice. Until you do, the exported analysis layer contains a review-required marker instead of that sentence. Do not claim that a word-level replacement preserves a clinical diagnosis without checking context.

In [ ]:
STIGMA_TERMS = ["substance abuse", "drug abuse", "abuser", "addict", "junkie", "dirty urine", "clean urine"]
def stigma_audit(text):
    return [term for term in STIGMA_TERMS if re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", str(text), re.I)]
notes["stigma_flags"] = notes["text"].map(stigma_audit)
display(notes.loc[notes["stigma_flags"].str.len().gt(0), ["note_id", "text", "stigma_flags"]])
# Preserve raw notes in memory. Edit the response below; no automatic rewrite is applied.
reviewed_phrase = "[Write the complete, respectful replacement sentence for the flagged note.]"
revision_rationale = "[Explain the original term, why you revised it, and why the meaning is preserved.]"
notes["analysis_safe_text"] = notes["normalized_text"]
flagged = notes["stigma_flags"].str.len().gt(0)
if "[" in reviewed_phrase:
    notes.loc[flagged, "analysis_safe_text"] = "[REVIEW REQUIRED: wording flagged]"
else:
    notes.loc[flagged, "analysis_safe_text"] = reviewed_phrase
print("Flagged text for the exported analysis layer:")
display(notes.loc[flagged, ["note_id", "analysis_safe_text"]])

In [ ]:
bias_response = """[How could a text model or an embedding reproduce stigma? Name one human review step before publishing labels.]"""
print(bias_response)

## 7. Select your pathway deliverable

### Foundational (Rise 4.31)
Write **about 200 words** on one terminology drift case that surprised you. Explain what changed, what evidence you checked, what you would **not** combine, and how your crosswalk records that limit. This reflection is required alongside the completed notebook.

### Applied (Rise 4.32)
Implement your own ingestion and **transformer-based NER** with a model appropriate to your chosen text. Keep any sensitive source out of this public Colab. Document model name/version and what mentions it misses. Build source-specific HEAL CDE/RxNorm decisions, run an **embedding-neighbor audit for stigma terms** (record model, query terms, neighbors, and your interpretation), then write a **500-word memo** on validity threats and mitigations. The provided notes can be a starting point, but Rise asks Applied learners to choose appropriate sources and build their own pipeline. The cell below gives slots for your code and audit; it does not substitute for implementing those stages.

In [ ]:
foundational_reflection = """[Foundational: replace with your ~200-word reflection on one drift case.]"""
applied_memo = """[Applied only: replace with your ~500-word memo on validity threats and mitigation.]"""
if PATHWAY == "Foundational":
    print("Reflection words:", len(foundational_reflection.split()))
else:
    print("Validity memo words:", len(applied_memo.split()))

In [ ]:
# Applied pathway only. Fill these in and add code cells for your own stages.
APPLIED_NER_MODEL = ""  # e.g., a verified transformer model suited to your chosen source
applied_ner_code_and_findings = """[Describe/link your implemented ingestion, transformer NER, and concept linking cells.]"""
embedding_audit_method_and_findings = """[Record model/version, stigma queries, nearest neighbors, interpretation, and safeguards.]"""
if PATHWAY == "Applied":
    print("TO COMPLETE: implement the model and audit above; document the results in this notebook.")

## 8. Export and review your work (Rise 4.4, steps 9–10)

The cell creates a crosswalk CSV with the exact columns in the Rise template, a derived analysis CSV, a README, metadata and a draft JSON schema. It **does not export raw note text**. The license field is intentionally blank until you confirm reuse permission. For Foundational, submit the **completed notebook and 200-word reflection** as Rise requests (the reflection also appears in the notebook/export). For Applied, Rise requires the notebook, crosswalk bundle including a completed schema/README/license, and the 500-word memo. A ZIP is provided for easy download; review it before sharing.

In [ ]:
status_options = {"supported", "uncertain", "no suitable mapping"}
if "[" in reviewed_phrase or "[" in revision_rationale:
    print("TO COMPLETE: revise the flagged wording and explain your change.")
if PATHWAY == "Foundational":
    if len(crosswalk) != 3 or crosswalk["mapping_status"].isin(status_options).sum() != 3:
        print("TO COMPLETE: fill all three crosswalk rows, including a valid mapping_status.")
    if any("[" in str(x) for x in crosswalk.to_numpy().ravel()):
        print("TO COMPLETE: replace bracketed crosswalk entries with evidence or an explicit 'not applicable'.")
    if "[" in foundational_reflection or not 170 <= len(foundational_reflection.split()) <= 230:
        print("TO COMPLETE: write a ~200-word Foundational reflection.")
else:
    if not APPLIED_NER_MODEL or "[" in embedding_audit_method_and_findings:
        print("TO COMPLETE: document your transformer NER and embedding-neighbor audit.")
    if "[" in applied_memo or not 450 <= len(applied_memo.split()) <= 550:
        print("TO COMPLETE: write a ~500-word Applied validity memo.")

# Keep original text in memory; export derived, synthetic practice output only.
analysis_cols = ["note_id", "analysis_safe_text", "drug_mentions", "dose_mentions", "route_mentions", "disorder_mentions"]
analysis = notes[analysis_cols].copy()
analysis["stigma_flag"] = notes["stigma_flags"].map(bool)
analysis.to_csv(OUT / "normalized_analysis_layer.csv", index=False)
crosswalk.to_csv(OUT / "terminology_crosswalk.csv", index=False)
answers = {
    "deidentification": deid_response,
    "normalization": normalization_response,
    "recognition": ner_response,
    "concept_linking": mapping_response,
    "reviewed_phrase": reviewed_phrase,
    "revision_rationale": revision_rationale,
    "embedding_bias": bias_response,
    "pathway_reflection_or_memo": foundational_reflection if PATHWAY == "Foundational" else applied_memo,
    "applied_ner_findings": applied_ner_code_and_findings if PATHWAY == "Applied" else "Not applicable",
    "embedding_audit": embedding_audit_method_and_findings if PATHWAY == "Applied" else "Not required for Foundational",
}
(OUT / "responses.md").write_text("\n\n".join(f"## {key.replace('_', ' ').title()}\n\n{value}" for key,value in answers.items()) + "\n", encoding="utf-8")
metadata = {"module": "WISER Module 4", "pathway": PATHWAY, "source_url": DATA_URL,
            "source_rows": len(all_notes), "representative_texts": len(notes),
            "selection": "first note_id per distinct text", "crosswalk_version": "draft-01",
            "raw_text_exported": False, "mapping_status_values": sorted(status_options)}
(OUT / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
license_text = "[Applied: insert the verified license and attribution for this bundle and every source. Do not assume clinical text can be redistributed.]"
(OUT / "LICENSE.txt").write_text(license_text + "\n", encoding="utf-8")
schema = {"title": "WISER Module 4 terminology crosswalk", "version": "draft-01", "type": "object",
          "required": ["mapping_id", "source_dataset", "original_wording", "mapping_status", "source_reference"],
          "properties": {col: {"type": "string"} for col in CROSSWALK_COLUMNS}}
schema["properties"]["mapping_status"]["enum"] = sorted(status_options)
(OUT / "crosswalk_schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
(OUT / "README.md").write_text(
    f"# WISER Module 4, {PATHWAY} pathway\n\n"
    f"Input: synthetic notes.csv at {DATA_URL}. {len(all_notes)} records contain {len(notes)} distinct texts; "
    "the practice pipeline keeps the first note_id per distinct text. Do not treat duplicate text as independent evidence.\n\n"
    "terminology_crosswalk.csv: learner decisions and source citations. normalized_analysis_layer.csv: reviewed display text and rule-based entity mentions; flagged wording is withheld until revised. "
    "responses.md: written review and reflection/memo. metadata.json: provenance. crosswalk_schema.json: field names/types. "
    "LICENSE.txt: requires a verified license for Applied reuse.\n\n"
    "Confirm the crosswalk against the Rise NSDUH teaching subset/codebooks and target definitions. "
    "No verified HEAL CDE or RxNorm concept identifiers are supplied by this notebook.\n",
    encoding="utf-8")
archive = shutil.make_archive("WISER_Module_4_submission", "zip", root_dir=".", base_dir=str(OUT))
print("Saved:", archive)
print("Review before submitting:", sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Outside Colab: download the ZIP from the file browser.")